In [31]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append('..')

import utilities.functions as functions

from utilities.functions import (
    load_data,
    check_key_uniqueness,
    merge_df,
    load_orders,
    process_orders_pandas
)

In [2]:
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 



Load the data --- se necessario salvar em stage - neste momento o estara comentado

In [3]:
URL_CONSUMER = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz"
df_consumer = load_data(URL_CONSUMER)[["customer_id", "active","created_at"]]

In [4]:
URL_RESTAURANT ="https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz"
df_restaurant= load_data(URL_RESTAURANT)

In [5]:
ab_test_url = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz"
df_ab = load_data(ab_test_url)

In [6]:
#save_parquet(df_consumer, "stage", "df_consumer.parquet")
#save_parquet(df_restaurant, "stage", "df_restaurant.parquet")
#save_parquet(df_consumer, "stage", "df_consumer.parquet")

Bronze layer - verify duplicates e nulo. e se necessario remover

In [7]:
check_key_uniqueness(df_consumer, ["customer_id","active","created_at"])

✅ Colunas ['customer_id', 'active', 'created_at'] são NOT NULL e UNIQUE.


(True, None, None, None)

In [8]:
check_key_uniqueness(df_restaurant, ["id"])

✅ Colunas ['id'] são NOT NULL e UNIQUE.


(True, None, None, None)

In [9]:
check_key_uniqueness(df_ab, ["customer_id","is_target"])
print(df_ab[df_ab["customer_id"].isna()])


❌ Colunas ['customer_id', 'is_target'] contêm valores nulos.

Soma de nulos por coluna:
customer_id    1
is_target      0
dtype: int64

Índices com nulos:
[81149]
      customer_id is_target
81149         NaN    target


In [10]:
df_ab_np = df_ab[~df_ab["customer_id"].isna()]

Antes de carregar a base ordens sera definido o publico todal, e da base de ordem serao filtrados somentes os clientes elegiceis

In [14]:
amostra_aleatoria = df_ab.sample(n=1000, random_state=42)['customer_id']
amostra_aleatoria

446058    572f0522bc81326538a1f6b4f18e5c6f439a523d052526...
460890    1feddbf3b52c54fb08a4069b52a0e71dcb18d251fd63c6...
749506    ee7504346c13bbf5dc2a967da32ccf278c73f521b98528...
33167     6acaa629949551a39ec9aa24d87306f12fecbeefc4c761...
503199    fe2246338a6e65bf97c01f51e6e0b984c32b29933e4c5a...
                                ...                        
523590    89b632f97093db6a6242b2d7659582b94ec5a9ca705b4f...
207051    b12861db4e2cbf456302de6d0571ef478d62f2c2a78cea...
704030    cb7e2cf0ad1161197695c8cb9158d321797697daa7e7e5...
140165    896dee5c432e843a2b66ecbbe03d66ee1629c490504eb2...
179195    a2a54cc2d10d5fac644ae23180067125937374335f5b1f...
Name: customer_id, Length: 1000, dtype: object

id=['fffe7b38b14ac2fca8906500fbecf87f5f8470179ed7ffa974892a3a15650604',
       'fffd6a19e4affba4589945ba2fe76804f25ad9301c0d3011219766b51106c2fe',
       'fffad85994233d99370bfb55ff196c7e82af11dd9825b121fa5f5563c2666c2a',
       '00086dd0b93a9c96d2d13e1b245bb82abbea957306b193be39375bdd853811f9',
       '00070a129efd4c5ffe3dfbc2ca704a7f891fd8b0ab4159930b813c541194c0cc',
       '0000c21984ae00cefb5d4931bfa49483dde546413c9b40c4228220f27d7ecdf2'
       ]
df_ab_np=df_ab_np[df_ab_np['customer_id'].isin(id)]

In [16]:
df_ab_np=df_ab_np[df_ab_np['customer_id'].isin(amostra_aleatoria)]
df_ab_np.head()

,customer_id,is_target
1442,f9d08cb722a2e6e8de241c903ea7118758d1eca1194af4...,target
1819,956b232489036fcb4e614db34c0768e6b514b967a6f412...,control
1929,6206f4b585fdd39a5a14bdbac75442a0cb728234d735cd...,target
4541,a12a6ad34995bd073c868832ea72712f1c3b9f4e1b1abd...,control
4843,5b2936cba781e3b0e563fbf4963bfda63a8039dd1935d4...,target


In [17]:
df_publico=merge_df(df_ab_np,df_consumer,['customer_id'],'inner')

In [18]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

,active,is_target,numero_clientes_distintos
0,False,control,1
1,False,target,2
2,True,control,446
3,True,target,550


In [19]:
df_publico = df_publico.dropna(subset=['active'])

In [20]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

,active,is_target,numero_clientes_distintos
0,False,control,1
1,False,target,2
2,True,control,446
3,True,target,550


Publico definido e todos os clientes marcados no teste a/b e existentes na base de clientes

Da base de ordens serao filtrados todos os clientes com orden nos meses de de dezembro e janeiro

In [21]:
URL_ORDERS = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz"

COLUMNS_TO_DROP = [
    'cpf','customer_name','delivery_address_city','delivery_address_country',
    'delivery_address_district','delivery_address_external_id',
    'delivery_address_latitude','delivery_address_longitude',
    'delivery_address_state','delivery_address_zip_code','items',
    'merchant_latitude','merchant_longitude','merchant_timezone',
    'order_scheduled','order_scheduled_date'
]

customer_ids = df_ab["customer_id"].astype(str).unique()

df_orders = load_orders(
    url=URL_ORDERS,
    customer_ids=customer_ids,
    columns_to_drop=COLUMNS_TO_DROP
)

df_orders.head()


,customer_id,merchant_id,order_created_at,order_id,order_total_amount,origin_platform
0,7ba88a68bb2a3504c6bd37a707af57a0b8d6e110a551c7...,a992a079a651e699d9149423761df2427c0e3af0a2a1b5...,2019-01-17T22:50:06.000Z,33e0612d62e5eb42aba15b58413137e441fbe906de2feb...,46.0,ANDROID
1,078acecdcf7fa89d356bfa349f14a8219db1ee161ce28a...,5152f28ee0518b8803ccf0a4096eb2ff8b81e9491861c9...,2019-01-17T17:51:26.000Z,148c4353a2952f3fe7973547283265eb22b575fb712ed2...,104.5,ANDROID
2,0e38a3237b5946e8ab2367b4f1a3ae6e77f1e215bc760c...,b6096419455c35d06105a5ef0d25c51f9dd40e1e99ac33...,2019-01-17T22:53:47.000Z,c37e495a91b498bb7b70a9e09ac115d0cdd443f152dc11...,35.0,IOS
3,cab1a004b7206d07910092c515a79834fea0a03d7d9054...,082bfdcdf6ccdc343e3c4d25ee376b5b6ca7e96ad8b04e...,2019-01-17T23:56:53.000Z,b4df94142d21354611247da9ca94f870c09b93989b531a...,40.8,IOS
4,aa7edf5b166b8c843aec3b96dc561222888734f3879123...,d7adb764bac29ccb77fb8f746ffbd531bf05ec30a7e130...,2019-01-17T23:40:53.000Z,4ff64b33b272c1886df21b63272220af6a82d1667dba70...,48.5,ANDROID


In [22]:
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id"]))
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id","order_created_at"]))

❌ Colunas ['customer_id', 'merchant_id', 'order_id'] possuem duplicações.
(False,                                                customer_id  \
1192509  c94d0872922e87c7f23669afcce3cc700d82ed75080a59...   
1192510  1e2af3429ee49b319a095258707980a7f1da2c2f1b6ca9...   
1192511  9a68122c178c32840eef9421530a375629d586ef9a7643...   
1192512  afdf6cab94f0e58e8a03940bb23243dc73263ad217205c...   
1192513  e9f79cc65b905e1e8cae0687605be386bc0275f6ec46ad...   
...                                                    ...   
3662316  648ae0e610811af0fccbe557b9a63a55c6e46adeeceb0b...   
3662317  5cab7f42316c5815d151d1fd0eebaecf9e6e53681257f0...   
3662318  1e91e110ba83f466ddbdb8ea448940e39e4e5ce16925e5...   
3662319  588becd71bc59b9a17ffcafe5823ce77777f308296f6f0...   
3662320  50862fb1670635160c98cd292768893dd65df05e5ae38e...   

                                               merchant_id  \
1192509  7e1b40dd1d08526b316dbbc60be12806ef7a5b7dd8278e...   
1192510  d1b5d9f45379a82adab72900747e635d4d9ee40d

Add orders para a base de publico

In [23]:
df_publico_orders=merge_df(df_publico,df_orders,['customer_id'],'inner')

In [25]:
df_publico_orders

,customer_id,is_target,active,created_at,merchant_id,order_created_at,order_id,order_total_amount,origin_platform
0,f9d08cb722a2e6e8de241c903ea7118758d1eca1194af4...,target,True,2018-01-05T13:13:31.034Z,a16bfd800d1f8dde449b7c223816f3edbce23c20d24602...,2019-01-13T22:25:18.000Z,fdbfd89677ecde53467c79431247dd9cae723cc772d408...,64.80,ANDROID
1,f9d08cb722a2e6e8de241c903ea7118758d1eca1194af4...,target,True,2018-01-05T13:13:31.034Z,a4617ad162318b58a3fea654dd18949f68051666e45fed...,2019-01-15T13:38:47.000Z,09895b62aab1f78e9290da00ada41c077e07c7ef040b15...,50.00,ANDROID
2,f9d08cb722a2e6e8de241c903ea7118758d1eca1194af4...,target,True,2018-01-05T13:13:31.034Z,a16bfd800d1f8dde449b7c223816f3edbce23c20d24602...,2019-01-27T22:50:09.000Z,d069e1819d3207d0de735e9ecbc683827b095d92bfce5a...,75.80,ANDROID
3,f9d08cb722a2e6e8de241c903ea7118758d1eca1194af4...,target,True,2018-01-05T13:13:31.034Z,a16bfd800d1f8dde449b7c223816f3edbce23c20d24602...,2018-12-28T22:50:09.000Z,d069e1819d3207d0de735e9ecbc683827b095d92bfce5a...,75.80,ANDROID
4,956b232489036fcb4e614db34c0768e6b514b967a6f412...,control,True,2018-01-03T15:36:59.020Z,c5ce36bf91a7d5d23d0e95ecbe1f6562b1ad0805409a5a...,2019-01-30T22:12:13.000Z,6eaf536e23815741a4a3792af738bb7bdfbb83b2bbd252...,71.80,DESKTOP
...,...,...,...,...,...,...,...,...,...
4623,8e98ccbc815a988b83a60b9ee8d292be709f5610014ea9...,target,True,2018-03-11T22:43:30.658Z,c5ce36bf91a7d5d23d0e95ecbe1f6562b1ad0805409a5a...,2019-01-30T23:32:35.000Z,e3ba0003f8b7c95dfb1f2c00dc249c1f01540dd2d2b023...,69.90,IOS
4624,48f84f3dc467fa6d7627966978de412a55ffbafb07fd03...,target,True,2018-04-05T13:34:26.380Z,5d4399f645d4121824a5450e099e1626d6310d1b31329b...,2019-01-22T21:10:24.000Z,6a5103bd966b8949eeed347bfdf0b186712a1dc08e2bbf...,34.80,IOS
4625,e2edefda6b897bec2bb116f269aeadac0673baadab47ce...,control,True,2018-04-06T04:02:48.994Z,2d2ddc276827f70caf6ac7192f2c83282644719f4ddf78...,2019-01-22T03:30:57.000Z,ac8f5db8967a44b32ca663ea2308120e1b8d0f77a69424...,28.99,ANDROID
4626,146265283d6da415aa33d64ba7e7a77a4201ac3786e519...,target,True,2018-01-11T22:55:49.552Z,166d642fbcfa8d603cd06c5247876cf2f3020495d0f0ed...,2019-01-11T00:29:32.000Z,f3b43cc8882dd73b1eac7a55ae8e1441e59dd5b25fe075...,19.90,ANDROID


Construcao de chave unica, e sumarizacoes visao cliente

In [ ]:
df_publico_orders[df_publico_orders['customer_id']=='fffe7b38b14ac2fca8906500fbecf87f5f8470179ed7ffa974892a3a15650604']

In [ ]:
df_publico_orders = pd.read_parquet(BASE_PATH / "gold" / "df_publico_orders.parquet")

In [27]:
df_publico=process_orders_pandas(df_publico_orders)

In [28]:
df_publico


,customer_id,is_target,active,created_at,merchant_id,order_created_at,order_total_amount,origin_platform,order_created_month,unique_order_hash,total_amount_mes,ticket_medio,num_pedidos_mes,num_pedidos_hist,prev_order_time,diff_days
2183,0005872d7c065b1b8afa4c4d8fbef264f992caa1022fd9...,target,True,2018-04-06T03:04:33.335Z,f77ea6efe1fd23b4b5e20410a246bb7cbedaad276bf7c7...,2018-12-09 01:09:47+00:00,21.0,ANDROID,12,18277965094323387885,68.9,22.966667,3,8,NaT,NaN
2184,0005872d7c065b1b8afa4c4d8fbef264f992caa1022fd9...,target,True,2018-04-06T03:04:33.335Z,05cc36ff4197b1cf16fc21544b54f44770e28457d5f5b0...,2018-12-16 00:36:17+00:00,35.9,ANDROID,12,7557430672743522669,68.9,22.966667,3,8,2018-12-09 01:09:47+00:00,6.0
2182,0005872d7c065b1b8afa4c4d8fbef264f992caa1022fd9...,target,True,2018-04-06T03:04:33.335Z,f1565604488284df8b4c5738d5143f69ea517e0fdbb2a0...,2018-12-17 00:06:13+00:00,12.0,ANDROID,12,238457097328583656,68.9,22.966667,3,8,2018-12-16 00:36:17+00:00,0.0
2178,0005872d7c065b1b8afa4c4d8fbef264f992caa1022fd9...,target,True,2018-04-06T03:04:33.335Z,05cc36ff4197b1cf16fc21544b54f44770e28457d5f5b0...,2019-01-07 22:57:57+00:00,44.8,ANDROID,1,17395200831230791362,139.2,27.840000,5,8,2018-12-17 00:06:13+00:00,21.0
2180,0005872d7c065b1b8afa4c4d8fbef264f992caa1022fd9...,target,True,2018-04-06T03:04:33.335Z,f77ea6efe1fd23b4b5e20410a246bb7cbedaad276bf7c7...,2019-01-08 01:09:47+00:00,21.0,ANDROID,1,13938685612346079295,139.2,27.840000,5,8,2019-01-07 22:57:57+00:00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4158,fffb2563cae68ab46bdba19d24d1982c3e78fda9a350e4...,target,True,2018-01-07T22:33:49.227Z,bf0e73a521174c6a71028dd3a3b9dc9e0a7ef8fb8b314a...,2018-12-30 23:27:07+00:00,49.9,IOS,12,8783161136092467961,49.9,49.900000,1,5,NaT,NaN
4156,fffb2563cae68ab46bdba19d24d1982c3e78fda9a350e4...,target,True,2018-01-07T22:33:49.227Z,6aad0b1618fb725e322adac831800baa68c182b0e0140e...,2019-01-16 11:36:10+00:00,64.0,IOS,1,2877400830056007768,244.4,61.100000,4,5,2018-12-30 23:27:07+00:00,16.0
4154,fffb2563cae68ab46bdba19d24d1982c3e78fda9a350e4...,target,True,2018-01-07T22:33:49.227Z,2cc1da7506c96275e9e5cb4bb407019f3854a24f3e7764...,2019-01-29 19:47:50+00:00,40.7,IOS,1,13620799853789124669,244.4,61.100000,4,5,2019-01-16 11:36:10+00:00,13.0
4157,fffb2563cae68ab46bdba19d24d1982c3e78fda9a350e4...,target,True,2018-01-07T22:33:49.227Z,bf0e73a521174c6a71028dd3a3b9dc9e0a7ef8fb8b314a...,2019-01-29 23:27:07+00:00,49.9,IOS,1,5292737110603922808,244.4,61.100000,4,5,2019-01-29 19:47:50+00:00,0.0


In [ ]:
#save_parquet(df_publico, "silver", "df_publico.parquet")

In [ ]:
#df_publico_orders[df_publico_orders['customer_id']=='fffe7bx38b14ac2fca8906500fbecf87f5f8470179ed7ffa974892a3a15650604']

Uma linha por cliente, com as variaveis necessarias

In [ ]:
#df_publico[df_publico['customer_id']=='fffe7b38b14ac2fca8906500fbecf87f5f8470179ed7ffa974892a3a15650604']

In [ ]:
#df_publico['customer_id'].unique()

In [34]:
df_pub_un = df_publico[["customer_id","is_target", "order_created_month", "num_pedidos_mes", "num_pedidos_hist",'total_amount_mes','ticket_medio']].drop_duplicates().reset_index(drop=True)
df_pub_un['pedidos_sum'] = np.where(
        df_pub_un['num_pedidos_mes'] > 10, '10+',
        df_pub_un['num_pedidos_mes'].astype(str)
    )
df_pub_un.head()

,customer_id,is_target,order_created_month,num_pedidos_mes,num_pedidos_hist,total_amount_mes,ticket_medio,pedidos_sum
0,0005872d7c065b1b8afa4c4d8fbef264f992caa1022fd9...,target,12,3,8,68.9,22.966667,3
1,0005872d7c065b1b8afa4c4d8fbef264f992caa1022fd9...,target,1,5,8,139.2,27.840000,5
2,0049a9cd42b5a234cbcc790556fa8c48b3d0d44353ac54...,target,1,7,7,172.1,24.585714,7
3,005f54ebf0a9afaaa6189281dc7f4e335ce71334cc63b5...,control,12,2,6,143.8,71.900000,2
4,005f54ebf0a9afaaa6189281dc7f4e335ce71334cc63b5...,control,1,4,6,199.3,49.825000,4


Salvar base visao cliente em gold layer

In [ ]:
df_pub_un.to_parquet(BASE_PATH / "gold" / "df_pub_un.parquet", index=False)

In [35]:
df_stats_mes = df_pub_un.groupby(['pedidos_sum', 'order_created_month']).agg(
    total_clientes=('customer_id', 'nunique')
)

df_stats_mes['pct_total_mes'] = (
    df_stats_mes['total_clientes'] /
    df_stats_mes.groupby('order_created_month')['total_clientes'].transform('sum') * 100
).round(2)

df_stats_mes.head(20)


total_clientes  pct_total_mes
pedidos_sum order_created_month                               
1           1                               446          44.64
            12                              367          53.27
10          1                                10           1.00
10+         1                                48           4.80
            12                               10           1.45
2           1                               191          19.12
            12                              141          20.46
3           1                               108          10.81
            12                               74          10.74
4           1                                78           7.81
            12                               37           5.37
5           1                                40           4.00
            12                               18           2.61
6           1                                26           2.60
            12                               21           3.05
7           1                                27           2.70
            12                               10           1.45
8           1                                12           1.20
            12                                7           1.02
9           1                                13           1.30

In [ ]:
df_pub_hist = df_publico[["customer_id","is_target",  "num_pedidos_hist"]].drop_duplicates().reset_index(drop=True)

,customer_id,is_target,num_pedidos_hist
0,0005872d7c065b1b8afa4c4d8fbef264f992caa1022fd9...,target,8
1,0049a9cd42b5a234cbcc790556fa8c48b3d0d44353ac54...,target,7
2,005f54ebf0a9afaaa6189281dc7f4e335ce71334cc63b5...,control,6
3,00624553890445cb4de900638fecde5911eed39457e652...,control,4
4,009e7e2735e8312bf5a0f36229bb50f05c7f5d472c1eb8...,control,8
...,...,...,...
994,ff6c94d1124a03cce294a81c229b86245e896ea84af449...,control,4
995,ff7836061c151ee1f0dcb9f83a7731f9efbcc651d41512...,control,1
996,ff95d6cdc3e16926d9724218d27af3080ec76d7d7f526d...,control,5
997,fff94f356582d0639bed0fe53df8bc7d04dc7a163b6915...,target,2


In [37]:
df_stats_hist = df_pub_hist.groupby(['num_pedidos_hist']).agg(
    total_clientes=('customer_id', 'nunique')  
).round(2)

df_stats_hist['pct_total'] = (df_stats_hist['total_clientes'] / df_stats_hist['total_clientes'].sum() * 100).round(2)

df_stats_hist.head(20)

,total_clientes,pct_total
num_pedidos_hist,,
1,228,22.82
2,270,27.03
3,85,8.51
4,126,12.61
5,47,4.70
6,51,5.11
7,30,3.00
8,28,2.80
9,25,2.50
